In [ ]:
# Выполнить до импортов. PyTorch из Colab не переустанавливаем.
%pip install -q "transformers==4.57.1" "accelerate>=0.34,<2" "datasets>=3,<5" "scikit-learn>=1.5,<2" "pymorphy3>=2,<3" "openai>=1,<3" matplotlib pandas


In [ ]:
from pathlib import Path
import os
import sys
import gc
import re
import json
import time
import pickle
import hashlib
import warnings
import random
from datetime import datetime, timezone
from functools import lru_cache
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import f1_score, mean_absolute_error, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42
N_SPLITS = 5
LEVEL_NAMES = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
LABELS = np.arange(1, 7)
LEVEL_TO_NUMBER = dict(zip(LEVEL_NAMES, LABELS.tolist()))
TARGET_COLUMN = 'median_rating_round'


DATA_DIR = Path(os.environ.get('CEFR_DATA_DIR', 'data'))
TEXTBOOK_PATH = DATA_DIR / 'ru_cefr_short(2).csv'
TARGET_PATH = DATA_DIR / 'target.csv'
VOCAB_PATHS = {level: DATA_DIR / f'new_vocab_{level.lower()}.txt' for level in LEVEL_NAMES[:5]}

RUN_VOCAB = os.environ.get('CEFR_RUN_VOCAB') == '1'                 # также запускает логистическую регрессию
RUN_TRANSFORMERS = os.environ.get('CEFR_RUN_TRANSFORMERS') == '1'
RUN_DEEPSEEK = True
# По умолчанию пересчитываем сохранённые ответы, без новых запросов к API.
DEEPSEEK_MODE = 'saved'          # 'saved' или 'api'
DEEPSEEK_SAVED_PATH = DATA_DIR / 'deepseek-chat_promptvocab2_results_rjl_raw.pickle'
# Для старого pickle порядок текстов проверяется по его исходному CSV.
DEEPSEEK_SOURCE_PATH = DATA_DIR / 'cefr_rjl.csv'
PROMPT_PATH = DATA_DIR / 'cefr_prompt2_20.txt'
DEEPSEEK_MODEL = 'deepseek-chat'

MODEL_SPECS = [
    {'tag': 'ruroberta_weighted', 'name': 'ai-forever/ruRoberta-large',
     'weighted': True, 'modes': ['textbooks', 'cv', 'cv_plus_textbooks']},
    {'tag': 'rubert_unweighted', 'name': 'ai-forever/ruBert-base',
     'weighted': False, 'modes': ['textbooks', 'cv']},
]
MAX_LENGTH = 256
EPOCHS = 5
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16           # уменьшите при нехватке GPU-памяти
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

# Дополнительный пересчёт старых CSV. Отсутствующие файлы будут явно отмечены.
# Здесь не выводим настройки обучения из имени: они неизвестны без журнала запуска.
LEGACY_CSV_PATHS = [
    DATA_DIR / 'predictions_cv_ruroberta_cross.csv',
    DATA_DIR / 'predictions_train_textbooks_ruroberta.csv',
    DATA_DIR / 'predictions_train_textbooks_rubert.csv',
    DATA_DIR / 'predictions_cv_rubert_cross.csv',
    DATA_DIR / 'predictions_cv_rubert_cross+textbooks.csv',
]

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = DATA_DIR / 'cefr_results' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Результаты:', OUTPUT_DIR)


In [ ]:
# Проверяем все обязательные файлы до запуска экспериментов.
required = [TEXTBOOK_PATH, TARGET_PATH]
if RUN_VOCAB:
    required.extend(VOCAB_PATHS.values())
if RUN_DEEPSEEK:
    if DEEPSEEK_MODE == 'saved':
        required.append(DEEPSEEK_SAVED_PATH)
        if DEEPSEEK_SAVED_PATH.suffix.lower() != '.csv':
            required.append(DEEPSEEK_SOURCE_PATH)
    elif DEEPSEEK_MODE == 'api':
        required.append(PROMPT_PATH)
    else:
        raise ValueError('DEEPSEEK_MODE должен быть saved или api')
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Загрузите файлы или исправьте пути:\n' + '\n'.join(missing))
print('Обязательные входные файлы найдены')


In [ ]:
def validate_labels(values, name='labels'):
    try:
        arr = np.asarray(values, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'{name}: нужны числовые метки 1–6') from exc
    if arr.ndim != 1 or arr.size == 0:
        raise ValueError(f'{name}: нужен непустой одномерный массив')
    if not np.isfinite(arr).all() or not np.isin(arr, LABELS).all():
        raise ValueError(f'{name}: найдены пропуски или метки вне целых 1–6')
    return arr.astype(int)

def checked_pair(y_true, y_pred):
    yt = validate_labels(y_true, 'y_true')
    yp = validate_labels(y_pred, 'y_pred')
    if yt.shape != yp.shape:
        raise ValueError('Различается число истинных меток и прогнозов')
    return yt, yp

def relaxed_macro_f1(y_true, y_pred):
    yt, yp = checked_pair(y_true, y_pred)
    corrected = np.where(np.abs(yp - yt) <= 1, yt, yp)
    return float(f1_score(yt, corrected, labels=LABELS, average='macro', zero_division=0))

def evaluate(y_true, y_pred):
    yt, yp = checked_pair(y_true, y_pred)
    strict = float(f1_score(yt, yp, labels=LABELS, average='macro', zero_division=0))
    relaxed = relaxed_macro_f1(yt, yp)
    assert relaxed + 1e-12 >= strict, 'Нарушено relaxed F1 >= F1'
    return {'Macro F1': strict, 'Relaxed Macro F1': relaxed,
            'MAE': float(mean_absolute_error(yt, yp))}

# Регрессионные проверки метрики: идеальный ответ, соседняя ошибка, далёкая ошибка.
assert evaluate(LABELS, LABELS)['Relaxed Macro F1'] == 1.0
assert relaxed_macro_f1(LABELS, [2, 1, 4, 3, 6, 5]) == 1.0
assert relaxed_macro_f1(LABELS, [3, 4, 5, 6, 1, 2]) == 0.0
assert relaxed_macro_f1([1, 2], [1, 2]) == f1_score([1, 2], [1, 2], labels=LABELS, average='macro', zero_division=0)
rng = np.random.default_rng(SEED)
for _ in range(100):
    evaluate(rng.integers(1, 7, 100), rng.integers(1, 7, 100))
print('Проверки метрик пройдены')


In [ ]:
def read_dataset(path, text_col, label_col):
    frame = pd.read_csv(path)
    missing = {text_col, label_col} - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name}: отсутствуют колонки {missing}')
    texts = frame[text_col]
    valid_texts = texts.map(lambda x: isinstance(x, str) and bool(x.strip()))
    if not valid_texts.all():
        raise ValueError(f'{path.name}: пустые/нестроковые тексты в строках {frame.index[~valid_texts].tolist()[:20]}')
    labels = validate_labels(frame[label_col], label_col)
    return frame.reset_index(drop=True), texts.to_numpy(dtype=object), labels

textbook_df, textbook_texts, textbook_y = read_dataset(
    TEXTBOOK_PATH, 'fragment', 'textbook-assigned cefr level')
target_df, target_texts, target_y = read_dataset(TARGET_PATH, 'text', TARGET_COLUMN)
counts = pd.Series(target_y).value_counts().reindex(LABELS, fill_value=0)
if (counts < N_SPLITS).any():
    raise ValueError(f'Для {N_SPLITS} фолдов в каждом классе нужно минимум {N_SPLITS} текстов: {counts.to_dict()}')
display(pd.DataFrame({'level': LEVEL_NAMES, 'target_count': counts.to_numpy()}))

# Единственное разбиение, общее для всех новых CV-экспериментов.
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(target_texts, target_y))
fold_ids = np.full(len(target_y), -1, dtype=int)
for fold, (tr, te) in enumerate(folds, 1):
    assert len(np.intersect1d(tr, te)) == 0
    assert (fold_ids[te] == -1).all()
    fold_ids[te] = fold
assert (fold_ids > 0).all()

def normalized_text_key(text):
    return ' '.join(text.lower().split())

target_keys = {normalized_text_key(t) for t in target_texts}
textbook_keys = {normalized_text_key(t) for t in textbook_texts}
overlap = target_keys & textbook_keys
duplicates = len(target_texts) - len(target_keys)
print('Дубликаты внутри нашего корпуса:', duplicates, '; совпадения с учебниками:', len(overlap))
if duplicates or overlap:
    warnings.warn('Есть одинаковые тексты. Разбиение здесь по строкам; проверьте возможную утечку между обучением и тестом перед публикацией результатов.')

base_predictions = pd.DataFrame({
    'row_id': np.arange(len(target_y)), 'text': target_texts,
    'y_true': target_y, 'fold': fold_ids,
})
if 'id' in target_df:
    base_predictions['source_id'] = target_df['id'].to_numpy()
base_predictions.to_csv(OUTPUT_DIR / 'target_and_folds.csv', index=False)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

manifest = {
    'seed': SEED, 'n_splits': N_SPLITS, 'target_column': TARGET_COLUMN,
    'label_scale': '1..6; transformers internal 0..5',
    'relaxed_definition': 'replace prediction with truth when abs(error)<=1; macro F1 over 1..6',
    'textbooks_sha256': sha256_file(TEXTBOOK_PATH), 'target_sha256': sha256_file(TARGET_PATH),
    'models': MODEL_SPECS, 'epochs': EPOCHS, 'max_length': MAX_LENGTH,
    'learning_rate': LEARNING_RATE, 'train_batch_size': TRAIN_BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE, 'weight_decay': WEIGHT_DECAY,
    'versions': {p: version(p) for p in ['numpy', 'pandas', 'scikit-learn']},
}
def save_manifest():
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
save_manifest()
summary_rows = {}
run_status = {}


In [ ]:
def record_result(name, predictions, protocol, fold_scores=None):
    yt, yp = checked_pair(target_y, predictions)
    result = evaluate(yt, yp)
    output = base_predictions.copy()
    output['prediction'] = yp
    output['prediction_level'] = [LEVEL_NAMES[p - 1] for p in yp]
    output['prediction_relaxed'] = np.where(np.abs(yp - yt) <= 1, yt, yp)
    output.to_csv(OUTPUT_DIR / f'{name}_predictions.csv', index=False)
    report = classification_report(yt, yp, labels=LABELS, target_names=LEVEL_NAMES, zero_division=0, output_dict=True)
    pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f'{name}_classification_report.csv')
    summary_rows[name] = {'experiment': name, 'protocol': protocol, 'n': len(yt), **result}
    pd.DataFrame(summary_rows.values()).to_csv(OUTPUT_DIR / 'metrics_summary.csv', index=False)
    if fold_scores is not None:
        table = pd.DataFrame(fold_scores).set_index('fold')
        table.to_csv(OUTPUT_DIR / f'{name}_fold_metrics.csv')
        table.agg(['mean', 'std']).to_csv(OUTPUT_DIR / f'{name}_fold_mean_std.csv')
    cm = confusion_matrix(yt, yp, labels=LABELS)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=LEVEL_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{name}_confusion.png', dpi=150)
    plt.show()
    plt.close(fig)
    run_status[name] = 'completed'
    print(name, result)
    return result


In [ ]:
LEVEL_THRESHOLDS = {'A1': 0.03, 'A2': 0.05, 'B1': 0.07, 'B2': 0.10, 'C1': 0.10}

def run_vocab_experiments():
    import pymorphy3
    missing = [str(p) for p in VOCAB_PATHS.values() if not p.exists()]
    if missing:
        raise FileNotFoundError('Не найдены словари: ' + ', '.join(missing))
    vocabs = {level: {line.strip().lower() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()}
              for level, path in VOCAB_PATHS.items()}
    if any(not words for words in vocabs.values()):
        raise ValueError('Обнаружен пустой словарь')
    morph = pymorphy3.MorphAnalyzer()
    @lru_cache(maxsize=200000)
    def lemma(word):
        return morph.parse(word)[0].normal_form
    rows = []
    for row_id, text in enumerate(target_texts):
        lemmas = [lemma(word) for word in re.findall(r'[а-яё]+', text.lower())]
        if not lemmas:
            raise ValueError(f'Строка {row_id}: нет кириллических слов; словарный уровень не определён')
        row = {'row_id': row_id, 'text': text, 'tokens': len(lemmas), 'unique_lemmas': len(set(lemmas))}
        for mode, words in [('tokens', lemmas), ('types', set(lemmas))]:
            oov = {level: sum(word not in vocabs[level] for word in words) / len(words) for level in LEVEL_THRESHOLDS}
            level = next((level for level, threshold in LEVEL_THRESHOLDS.items() if oov[level] <= threshold), 'C2')
            row[f'level_{mode}'] = LEVEL_TO_NUMBER[level]
            row.update({f'{level}_{mode}': round(value * 100, 2) for level, value in oov.items()})
        rows.append(row)
    features = pd.DataFrame(rows)
    features.to_csv(OUTPUT_DIR / 'vocab_features.csv', index=False)
    manifest['vocab_sha256'] = {k: sha256_file(v) for k, v in VOCAB_PATHS.items()}
    manifest['vocab_thresholds'] = LEVEL_THRESHOLDS
    save_manifest()
    for mode in ['tokens', 'types']:
        record_result(f'vocab_{mode}', features[f'level_{mode}'], 'fixed_vocab_rules')
    x = features[[f'{level}_tokens' for level in LEVEL_THRESHOLDS]].to_numpy()
    oof = np.full(len(target_y), -1, dtype=int)
    scores = []
    for fold, (tr, te) in enumerate(folds, 1):
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
        model.fit(x[tr], target_y[tr])
        oof[te] = model.predict(x[te])
        scores.append({'fold': fold, **evaluate(target_y[te], oof[te])})
    record_result('logreg_vocab_cv5', oof, 'cv5_oof', scores)

if RUN_VOCAB:
    run_vocab_experiments()
else:
    run_status['vocab_and_logreg'] = 'disabled'


In [ ]:
def extract_cefr_level(value):
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, dict):
        return extract_cefr_level(value.get('level'))
    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value) if np.isfinite(value) and value in LABELS else None
    text = str(value).strip()
    if text.upper().startswith('ERROR'):
        return None
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.IGNORECASE).strip()
    try:
        parsed = json.loads(text)
    except (json.JSONDecodeError, TypeError):
        parsed = None
    if isinstance(parsed, dict):
        return extract_cefr_level(parsed.get('level'))
    if isinstance(parsed, (int, float)) and not isinstance(parsed, bool):
        return extract_cefr_level(parsed)
    if isinstance(parsed, str):
        text = parsed.strip()
    matches = set(re.findall(r'\b[ABC][12]\b', text.upper()))
    return LEVEL_TO_NUMBER[next(iter(matches))] if len(matches) == 1 else None

assert extract_cefr_level('{"level": "B1"}') == 3
assert extract_cefr_level('A2 or B1') is None
assert extract_cefr_level('There are 3 words') is None

class PlainDataUnpickler(pickle.Unpickler):
    # Старый raw.pickle — список строк. Загрузка объектов/классов не требуется.
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Разрешён только pickle с простыми данными, без классов')

def check_text_alignment(frame, name):
    if 'text' not in frame or len(frame) != len(target_texts):
        raise ValueError(f'{name}: требуется колонка text и ровно {len(target_texts)} строк')
    if not np.array_equal(frame['text'].to_numpy(), target_texts):
        raise ValueError(f'{name}: тексты или их порядок отличаются от нашего корпуса')

def load_saved_deepseek():
    if DEEPSEEK_SAVED_PATH.suffix.lower() == '.csv':
        frame = pd.read_csv(DEEPSEEK_SAVED_PATH)
        check_text_alignment(frame, DEEPSEEK_SAVED_PATH.name)
        col = next((c for c in ['raw_response', 'cefr_value', 'prediction'] if c in frame), None)
        if col is None:
            raise ValueError('В CSV нет колонки с ответами')
        values = frame[col].tolist()
    else:
        source = pd.read_csv(DEEPSEEK_SOURCE_PATH)
        check_text_alignment(source, DEEPSEEK_SOURCE_PATH.name)
        with DEEPSEEK_SAVED_PATH.open('rb') as f:
            values = PlainDataUnpickler(f).load()
        if not isinstance(values, (list, tuple)) or len(values) != len(target_texts):
            raise ValueError('Число сохранённых ответов не совпадает с числом текстов')
    manifest['deepseek_saved_sha256'] = sha256_file(DEEPSEEK_SAVED_PATH)
    save_manifest()
    return values

def query_deepseek():
    from getpass import getpass
    from openai import OpenAI
    prompt = PROMPT_PATH.read_text(encoding='utf-8')
    if '{TEXT}' not in prompt:
        raise ValueError('В промпте отсутствует {TEXT}')
    key = os.environ.get('DEEPSEEK_API_KEY')
    if not key and 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            key = userdata.get('DEEPSEEK_API_KEY')
        except Exception:
            pass
    if not key:
        key = getpass('DEEPSEEK_API_KEY: ')
    if not key:
        raise ValueError('Не задан ключ DeepSeek')
    client = OpenAI(api_key=key, base_url='https://api.deepseek.com/v1', max_retries=0, timeout=120)
    settings = {'model': DEEPSEEK_MODEL, 'temperature': 0.0, 'top_p': 0.8,
                'max_tokens': 20, 'frequency_penalty': 0.0, 'presence_penalty': 0.0}
    manifest['deepseek_api'] = {**settings, 'prompt_sha256': sha256_file(PROMPT_PATH)}
    save_manifest()
    (OUTPUT_DIR / 'deepseek_prompt.txt').write_text(prompt, encoding='utf-8')
    records = []
    raw_path = OUTPUT_DIR / 'deepseek_api_raw.jsonl'
    # Повторное выполнение этой ячейки в той же сессии продолжает текущий запуск.
    if raw_path.exists():
        records = [json.loads(line) for line in raw_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        for i, record in enumerate(records):
            if record['row_id'] != i or record['text'] != target_texts[i] or record['settings'] != manifest['deepseek_api']:
                raise ValueError('Кеш API не соответствует текущим данным/настройкам')
    for i in range(len(records), len(target_texts)):
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    messages=[{'role': 'user', 'content': prompt.replace('{TEXT}', target_texts[i])}], **settings)
                raw = response.choices[0].message.content or ''
                finish_reason = response.choices[0].finish_reason
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
        record = {'row_id': i, 'text': target_texts[i], 'raw_response': raw,
                  'finish_reason': finish_reason, 'settings': manifest['deepseek_api']}
        with raw_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
        records.append(record)
        if (i + 1) % 50 == 0:
            print('DeepSeek:', i + 1, '/', len(target_texts))
    return [r['raw_response'] for r in records]

if RUN_DEEPSEEK:
    if DEEPSEEK_MODE not in {'saved', 'api'}:
        raise ValueError('DEEPSEEK_MODE должен быть saved или api')
    raw_responses = load_saved_deepseek() if DEEPSEEK_MODE == 'saved' else query_deepseek()
    deepseek_predictions = [extract_cefr_level(r) for r in raw_responses]
    deepseek_table = base_predictions.copy()
    deepseek_table['raw_response'] = raw_responses
    deepseek_table['prediction'] = deepseek_predictions
    deepseek_table.to_csv(OUTPUT_DIR / 'deepseek_parsed.csv', index=False)
    bad = deepseek_table['prediction'].isna()
    if bad.any():
        deepseek_table.loc[bad].to_csv(OUTPUT_DIR / 'deepseek_unparsed.csv', index=False)
        raise ValueError(f'Не разобрано {bad.sum()} ответов. См. deepseek_unparsed.csv; метрика по неполному корпусу не рассчитана.')
    record_result('deepseek', deepseek_predictions, 'llm_' + DEEPSEEK_MODE)
else:
    run_status['deepseek'] = 'disabled'


In [ ]:
for path in LEGACY_CSV_PATHS:
    name = 'legacy_' + path.stem
    if not path.exists():
        run_status[name] = 'file_missing'
        print('Старый файл отсутствует:', path.name)
        continue
    try:
        legacy = pd.read_csv(path)
        check_text_alignment(legacy, path.name)
        if 'prediction' not in legacy:
            raise ValueError(f'{path.name}: нет колонки prediction')
        checked_pair(target_y, legacy['prediction'])
    except (ValueError, pd.errors.ParserError) as exc:
        run_status[name] = 'not_evaluated: ' + str(exc)
        print('Старые прогнозы не оценены:', exc)
        continue
    record_result(name, legacy['prediction'], 'legacy_saved_protocol_unverified')


In [ ]:
if RUN_TRANSFORMERS:
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
    from sklearn.utils.class_weight import compute_class_weight
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет; обучение на CPU будет медленным')
    manifest['versions'].update({p: version(p) for p in ['torch', 'transformers', 'datasets', 'accelerate']})
    save_manifest()

    class WeightedTrainer(Trainer):
        def __init__(self, *args, class_weights, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights
            self.model_accepts_loss_kwargs = False

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            inputs = dict(inputs)
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(
                outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device))
            return (loss, outputs) if return_outputs else loss

    def make_dataset(texts, labels, tokenizer):
        ds = Dataset.from_dict({'text': list(texts), 'labels': (validate_labels(labels) - 1).tolist()})
        ds = ds.map(lambda batch: tokenizer(batch['text'], padding='max_length', truncation=True,
                                            max_length=MAX_LENGTH), batched=True, remove_columns=['text'])
        # Сохраняем token_type_ids, если токенизатор их возвращает.
        ds.set_format(type='torch')
        return ds

    def train_and_predict(spec, train_ds, train_labels, test_ds, name):
        set_seed(SEED)
        model = AutoModelForSequenceClassification.from_pretrained(
            spec['name'], num_labels=6,
            id2label=dict(enumerate(LEVEL_NAMES)), label2id={name: i for i, name in enumerate(LEVEL_NAMES)})
        manifest.setdefault('model_revisions', {})[spec['name']] = getattr(model.config, '_commit_hash', None)
        save_manifest()
        args = TrainingArguments(
            output_dir=str(OUTPUT_DIR / 'training' / name), learning_rate=LEARNING_RATE,
            per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
            num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, save_strategy='no',
            report_to='none', seed=SEED, data_seed=SEED, logging_steps=20,
            dataloader_num_workers=0, fp16=False, bf16=False)
        kwargs = dict(model=model, args=args, train_dataset=train_ds)
        trainer = None
        try:
            if spec['weighted']:
                encoded = validate_labels(train_labels) - 1
                if set(encoded) != set(range(6)):
                    raise ValueError('В обучающей части отсутствуют классы для расчёта balanced weights')
                weights = compute_class_weight(class_weight='balanced', classes=np.arange(6), y=encoded)
                trainer = WeightedTrainer(**kwargs, class_weights=torch.tensor(weights, dtype=torch.float32))
            else:
                trainer = Trainer(**kwargs)
            trainer.train()
            logits = trainer.predict(test_ds).predictions
            if isinstance(logits, tuple):
                logits = logits[0]
            predictions = np.asarray(logits).argmax(axis=1) + 1
            pd.DataFrame(trainer.state.log_history).to_csv(OUTPUT_DIR / f'{name}_training_log.csv', index=False)
            return predictions
        finally:
            del trainer, model, kwargs
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def run_transformer_experiment(spec, mode):
        name = spec['tag'] + '_' + ('cv5' if mode == 'cv' else 'cv5_plus_textbooks' if mode == 'cv_plus_textbooks' else mode)
        tokenizer = AutoTokenizer.from_pretrained(spec['name'])
        target_ds = make_dataset(target_texts, target_y, tokenizer)
        if mode == 'textbooks':
            train_ds = make_dataset(textbook_texts, textbook_y, tokenizer)
            pred = train_and_predict(spec, train_ds, textbook_y, target_ds, name)
            return record_result(name, pred, 'textbooks_to_target')
        if mode not in {'cv', 'cv_plus_textbooks'}:
            raise ValueError(mode)
        from datasets import concatenate_datasets
        extra_ds = make_dataset(textbook_texts, textbook_y, tokenizer) if mode == 'cv_plus_textbooks' else None
        oof = np.full(len(target_y), -1, dtype=int)
        scores = []
        for fold, (tr, te) in enumerate(folds, 1):
            print(name, 'fold', fold, '/', N_SPLITS)
            train_ds = target_ds.select(tr.tolist())
            train_labels = target_y[tr]
            if extra_ds is not None:
                train_ds = concatenate_datasets([train_ds, extra_ds])
                train_labels = np.concatenate([train_labels, textbook_y])
            pred = train_and_predict(spec, train_ds, train_labels, target_ds.select(te.tolist()), f'{name}_fold{fold}')
            oof[te] = pred
            fold_output = base_predictions.iloc[te].copy()
            fold_output['prediction'] = pred
            fold_output.to_csv(OUTPUT_DIR / f'{name}_fold{fold}_predictions.csv', index=False)
            scores.append({'fold': fold, **evaluate(target_y[te], pred)})
        assert (oof >= 1).all(), 'Не для всех строк получены OOF-прогнозы'
        return record_result(name, oof, 'cv5_oof' if mode == 'cv' else 'cv5_oof_plus_textbooks', scores)
else:
    run_status['transformers'] = 'disabled'


In [ ]:
if RUN_TRANSFORMERS and 'textbooks' in MODEL_SPECS[0]['modes']:
    run_transformer_experiment(MODEL_SPECS[0], 'textbooks')


In [ ]:
if RUN_TRANSFORMERS and 'cv' in MODEL_SPECS[0]['modes']:
    run_transformer_experiment(MODEL_SPECS[0], 'cv')


In [ ]:
if RUN_TRANSFORMERS and 'cv_plus_textbooks' in MODEL_SPECS[0]['modes']:
    run_transformer_experiment(MODEL_SPECS[0], 'cv_plus_textbooks')


In [ ]:
if RUN_TRANSFORMERS and 'textbooks' in MODEL_SPECS[1]['modes']:
    run_transformer_experiment(MODEL_SPECS[1], 'textbooks')


In [ ]:
if RUN_TRANSFORMERS and 'cv' in MODEL_SPECS[1]['modes']:
    run_transformer_experiment(MODEL_SPECS[1], 'cv')


In [ ]:
summary = pd.DataFrame(summary_rows.values())
display(summary)
display(pd.DataFrame([{'experiment': k, 'status': v} for k, v in run_status.items()]))
(OUTPUT_DIR / 'run_status.json').write_text(json.dumps(run_status, ensure_ascii=False, indent=2), encoding='utf-8')
save_manifest()
print('Сводка завершённых экспериментов сохранена. Проверьте статусы выше. Файлы:', OUTPUT_DIR)
